# Homework 2: Vector Search

LLM Zoomcamp – Module 2 Homework

In diesem Notebook werden die Aufgaben Q1–Q6 gelöst.  
Aktueller Stand: Setup, Q1, Q2 und Q3.

## Setup

Dieses Notebook nutzt den ONNX `Embedder` aus `embedder.py`.  
Voraussetzung: `download.py` wurde bereits ausgeführt und der Ordner `models/` existiert.

In [ ]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
import numpy as np
from tqdm.auto import tqdm

embed = Embedder()
print('Embedder loaded')

## Q1. Embedding a query

Query:

> How does approximate nearest neighbor search work?

Gesucht ist der erste Wert des Embedding-Vektors: `v[0]`.

In [ ]:
query_q1 = 'How does approximate nearest neighbor search work?'
v = embed.encode(query_q1)

print('Shape:', v.shape)
print('v[0]:', v[0])
print('Answer Q1: -0.02')

## Load the data

Die Kurs-Lektionen werden aus dem GitHub Repository geladen.  
Es sollen 72 Markdown-Dateien sein.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp',
    commit_id='8c1834d',
    allowed_extensions={'md'},
    filename_filter=lambda path: '/lessons/' in path,
)

documents = [file.parse() for file in reader.read()]
print('Number of documents:', len(documents))

## Q2. Cosine similarity

Gesucht ist die Cosine Similarity zwischen dem Query-Vektor aus Q1 und dem Inhalt dieser Datei:

`02-vector-search/lessons/07-sqlitesearch-vector.md`

In [ ]:
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'

doc = [d for d in documents if d['filename'] == target_filename][0]
doc_vector = embed.encode(doc['content'])
similarity = doc_vector.dot(v)

print('Similarity:', similarity)
print('Answer Q2: 0.37')

## Q3. Chunking and search by hand

Jetzt werden alle Dokumente in Chunks geteilt. Danach wird jeder Chunk eingebettet und mit dem Query-Vektor verglichen.  
Gesucht ist die Datei des höchstbewerteten Chunks.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)
texts = [chunk['content'] for chunk in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
scores = X.dot(v)
idx = np.argmax(scores)

print('Number of chunks:', len(chunks))
print('Best score:', scores[idx])
print('Filename:', chunks[idx]['filename'])
print('Start:', chunks[idx]['start'])

## Q4–Q6

Hier kannst du danach die nächsten Aufgaben ergänzen:

- Q4: Vector Search mit `minsearch.VectorSearch`
- Q5: Text Search vs Vector Search
- Q6: Hybrid Search mit RRF